# Gold Layer - Star Schema (1 fact + 2 dimensions)

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.dim_station AS
SELECT DISTINCT
  station_id,
  operator,
  CAST(power_kw AS DOUBLE) AS station_power_kw,
  CASE
    WHEN CAST(power_kw AS DOUBLE) <= 11 THEN 'AC home'
    WHEN CAST(power_kw AS DOUBLE) <= 22 THEN 'AC destination'
    WHEN CAST(power_kw AS DOUBLE) <= 50 THEN 'DC fast'
    ELSE 'DC ultra-fast'
  END AS charger_class
FROM workspace.default.bronze_stations;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.dim_time AS
SELECT DISTINCT
  DATE(started_at)                          AS date_key,
  YEAR(started_at)                          AS year,
  QUARTER(started_at)                       AS quarter,
  MONTH(started_at)                         AS month,
  DAYOFWEEK(started_at)                     AS day_of_week,
  CASE WHEN DAYOFWEEK(started_at) IN (1,7) THEN TRUE ELSE FALSE END AS is_weekend,
  MONTHNAME(started_at)                     AS month_name
FROM workspace.default.silver_clean;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.fact_sessions AS
SELECT
  session_id,
  station_id,
  DATE(started_at) AS date_key,
  HOUR(started_at) AS hour_of_day,
  CASE WHEN HOUR(started_at) BETWEEN 13 AND 21 THEN 'day' ELSE 'night' END AS tariff_period,
  driver_id,
  session_type,
  started_at,
  ended_at,
  duration_minutes,
  energy_kwh,
  price_per_kwh,
  cost_pln
FROM workspace.default.silver_clean
WHERE anomaly_reason = 'ok';

In [0]:
%sql
SELECT 'fact_sessions' AS tbl, COUNT(*) AS rows FROM workspace.default.fact_sessions
UNION ALL SELECT 'dim_station', COUNT(*) FROM workspace.default.dim_station
UNION ALL SELECT 'dim_time', COUNT(*) FROM workspace.default.dim_time;